# 06 - Results and uncertainty

This notebook converts prediction-level artifacts into condition metrics, robustness summaries, defense effects, and uncertainty intervals. Metrics are calculated within each model seed before any across-seed averaging.

**Inputs:** Raw and defended prediction partitions.  
**Outputs:** Per-seed metrics, across-seed summaries, and unpaired and paired bootstrap intervals.

## Setup and metric helpers

Prediction files share one schema, allowing a single aggregation path for classical and Transformer systems. Deterministic bootstrap seeds are derived from the condition keys so rerunning the notebook reproduces each interval independently of file order.

In [1]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import accuracy_score, f1_score

# Same path setup as notebook 00; repeated for independent execution.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CONFIG = yaml.safe_load((ROOT / 'config/experiment.yaml').read_text(encoding='utf-8'))

PATHS = {key: ROOT / value for key, value in CONFIG['paths'].items()}
for directory in (PATHS['tables'],):
    directory.mkdir(parents=True, exist_ok=True)

# Same deterministic seed helper as notebook 02.
def stable_int(*parts, modulo=2**32 - 1):
    digest = hashlib.sha256('||'.join(map(str, parts)).encode('utf-8')).hexdigest()
    return int(digest[:16], 16) % modulo

def condition_metric_rows(frame):
    keys = ['dataset', 'model', 'model_seed', 'attack', 'strength', 'attack_seed', 'defense']
    clean = frame[frame['attack'] == 'clean'].set_index('example_id')['correct']
    rows = []
    for values, group in frame.groupby(keys, dropna=False, observed=True):
        row = dict(zip(keys, values))
        row.update({
            'n': len(group),
            'accuracy': accuracy_score(group['gold'], group['prediction']),
            'macro_f1': f1_score(group['gold'], group['prediction'], average='macro', zero_division=0),
            'coverage': float((group['changed_count'] > 0).mean()),
        })
        if row['attack'] == 'clean':
            row['attack_success_rate'] = 0.0
        else:
            aligned_clean = clean.reindex(group['example_id']).to_numpy(dtype=bool)
            attacked_correct = group['correct'].to_numpy(dtype=bool)
            denominator = aligned_clean.sum()
            row['attack_success_rate'] = float((aligned_clean & ~attacked_correct).sum() / denominator) if denominator else np.nan
        rows.append(row)
    return rows


## Condition scores and robustness summaries

Accuracy and Macro-F1 are computed for every dataset, model seed, attack, strength, seed, and defense. Attack success rate is the fraction of examples that were correct when clean but incorrect after attack, using only clean-correct examples as the denominator.

Robustness drop is `clean score − attacked score`. Defense recovery is `(defended attacked score − raw attacked score) / (raw clean score − raw attacked score)` and remains unclipped; clean delta records any defense effect on unperturbed text.

In [2]:
prediction_files = sorted(PATHS['predictions'].glob('**/*.parquet'))
if not prediction_files:
    raise FileNotFoundError('No prediction artifacts found; run notebooks 03--05 first')
seed_rows = []
for path in prediction_files:
    frame = pd.read_parquet(path)
    seed_rows.extend(condition_metric_rows(frame))

seed_metrics = pd.DataFrame(seed_rows)
seed_metrics.to_csv(PATHS['tables'] / 'condition_metrics_by_seed.csv', index=False)

group_keys = ['dataset', 'model', 'attack', 'strength', 'defense']
metric_columns = ['accuracy', 'macro_f1', 'attack_success_rate', 'coverage']
summary = seed_metrics.groupby(group_keys, as_index=False).agg(
    runs=('accuracy', 'size'),
    **{f'{metric}_mean': (metric, 'mean') for metric in metric_columns},
    **{f'{metric}_std': (metric, 'std') for metric in metric_columns},
)
summary[[column for column in summary if column.endswith('_std')]] = summary[[column for column in summary if column.endswith('_std')]].fillna(0.0)

clean = summary[summary['attack'] == 'clean'][['dataset', 'model', 'defense', 'accuracy_mean', 'macro_f1_mean']].rename(columns={'accuracy_mean': 'clean_accuracy', 'macro_f1_mean': 'clean_macro_f1'})
summary = summary.merge(clean, on=['dataset', 'model', 'defense'], how='left', validate='many_to_one')
summary['accuracy_drop'] = summary['clean_accuracy'] - summary['accuracy_mean']
summary['macro_f1_drop'] = summary['clean_macro_f1'] - summary['macro_f1_mean']

raw = summary[summary['defense'] == 'none'][['dataset', 'model', 'attack', 'strength', 'accuracy_mean', 'macro_f1_mean', 'clean_accuracy', 'clean_macro_f1']].rename(columns={'accuracy_mean': 'raw_attacked_accuracy', 'macro_f1_mean': 'raw_attacked_macro_f1', 'clean_accuracy': 'raw_clean_accuracy', 'clean_macro_f1': 'raw_clean_macro_f1'})
summary = summary.merge(raw, on=['dataset', 'model', 'attack', 'strength'], how='left', validate='many_to_one')
for metric in ['accuracy', 'macro_f1']:
    denominator = summary[f'raw_clean_{metric}'] - summary[f'raw_attacked_{metric}']
    numerator = summary[f'{metric}_mean'] - summary[f'raw_attacked_{metric}']
    summary[f'{metric}_defense_recovery'] = np.where(denominator > 0, numerator / denominator, np.nan)
    summary[f'{metric}_clean_delta'] = summary[f'clean_{metric}'] - summary[f'raw_clean_{metric}']
summary.to_csv(PATHS['tables'] / 'metrics_summary.csv', index=False)
summary.head()


,dataset,model,attack,strength,defense,runs,accuracy_mean,macro_f1_mean,attack_success_rate_mean,coverage_mean,...,accuracy_drop,macro_f1_drop,raw_attacked_accuracy,raw_attacked_macro_f1,raw_clean_accuracy,raw_clean_macro_f1,accuracy_defense_recovery,accuracy_clean_delta,macro_f1_defense_recovery,macro_f1_clean_delta
0,banking77,bpe_tfidf,clean,0.0,nfc,1,0.882143,0.882234,0.0,0.0,...,0.0,0.0,0.882143,0.882234,0.882143,0.882234,NaN,0.0,NaN,0.0
1,banking77,bpe_tfidf,clean,0.0,nfkc,1,0.882143,0.882234,0.0,0.0,...,0.0,0.0,0.882143,0.882234,0.882143,0.882234,NaN,0.0,NaN,0.0
2,banking77,bpe_tfidf,clean,0.0,nfkc_strip,1,0.882143,0.882234,0.0,0.0,...,0.0,0.0,0.882143,0.882234,0.882143,0.882234,NaN,0.0,NaN,0.0
3,banking77,bpe_tfidf,clean,0.0,nfkc_strip_confusable,1,0.882143,0.882234,0.0,0.0,...,0.0,0.0,0.882143,0.882234,0.882143,0.882234,NaN,0.0,NaN,0.0
4,banking77,bpe_tfidf,clean,0.0,none,1,0.882143,0.882234,0.0,0.0,...,0.0,0.0,0.882143,0.882234,0.882143,0.882234,NaN,0.0,NaN,0.0


## Principal bootstrap intervals

For the clean condition and primary-strength attacks, examples are resampled with replacement within each condition. Percentile intervals are reported for accuracy and Macro-F1. These intervals quantify finite-test-set uncertainty and are separate from variation across Transformer training seeds.

In [ ]:
def bootstrap_interval(group, resamples, seed):
    rng = np.random.default_rng(seed)
    gold = group['gold'].to_numpy(dtype=int)
    prediction = group['prediction'].to_numpy(dtype=int)
    accuracy_values = np.empty(resamples)
    f1_values = np.empty(resamples)
    for idx in range(resamples):
        sample = rng.integers(0, len(group), len(group))
        accuracy_values[idx] = accuracy_score(gold[sample], prediction[sample])
        f1_values[idx] = f1_score(gold[sample], prediction[sample], average='macro', zero_division=0)
    return {
        'accuracy_ci_low': float(np.quantile(accuracy_values, 0.025)),
        'accuracy_ci_high': float(np.quantile(accuracy_values, 0.975)),
        'macro_f1_ci_low': float(np.quantile(f1_values, 0.025)),
        'macro_f1_ci_high': float(np.quantile(f1_values, 0.975)),
    }

primary_strength = float(CONFIG['analysis']['primary_strength'])
bootstrap_rows = []
for path in prediction_files:
    frame = pd.read_parquet(path)
    selected = frame[frame['defense'].isin(['none', 'nfkc_strip_confusable']) & ((frame['attack'] == 'clean') | (frame['strength'] == primary_strength))]
    keys = ['dataset', 'model', 'model_seed', 'attack', 'strength', 'attack_seed', 'defense']
    for values, group in selected.groupby(keys, observed=True):
        row = dict(zip(keys, values))
        row.update(bootstrap_interval(group, int(CONFIG['analysis']['bootstrap_resamples']), stable_int(CONFIG['analysis']['bootstrap_seed'], *values)))
        row['n'] = len(group)
        bootstrap_rows.append(row)
bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(PATHS['tables'] / 'principal_bootstrap_intervals.csv', index=False)


## Paired robustness-drop intervals

Clean and attacked predictions are aligned one-to-one by example ID and gold label before resampling. The same sampled IDs contribute to both scores in each replicate, preserving within-example dependence and directly estimating uncertainty in the clean-to-attacked drop.

In [ ]:
def paired_drop_interval(aligned, resamples, seed):
    rng = np.random.default_rng(seed)
    gold = aligned['gold'].to_numpy(dtype=int)
    clean_prediction = aligned['clean_prediction'].to_numpy(dtype=int)
    attacked_prediction = aligned['prediction'].to_numpy(dtype=int)
    accuracy_drop = np.empty(resamples)
    macro_f1_drop = np.empty(resamples)
    for idx in range(resamples):
        sample = rng.integers(0, len(aligned), len(aligned))
        sample_gold = gold[sample]
        accuracy_drop[idx] = accuracy_score(sample_gold, clean_prediction[sample]) - accuracy_score(sample_gold, attacked_prediction[sample])
        macro_f1_drop[idx] = f1_score(sample_gold, clean_prediction[sample], average='macro', zero_division=0) - f1_score(sample_gold, attacked_prediction[sample], average='macro', zero_division=0)
    return {
        'accuracy_drop_ci_low': float(np.quantile(accuracy_drop, 0.025)),
        'accuracy_drop_ci_high': float(np.quantile(accuracy_drop, 0.975)),
        'macro_f1_drop_ci_low': float(np.quantile(macro_f1_drop, 0.025)),
        'macro_f1_drop_ci_high': float(np.quantile(macro_f1_drop, 0.975)),
    }

paired_rows = []
for path in prediction_files:
    frame = pd.read_parquet(path)
    if not frame['defense'].iloc[0] in {'none', 'nfkc_strip_confusable'}:
        continue
    clean_frame = frame[frame['attack'] == 'clean'][['example_id', 'gold', 'prediction']].rename(columns={'prediction': 'clean_prediction'})
    attacked = frame[(frame['attack'] != 'clean') & (frame['strength'] == primary_strength)]
    keys = ['dataset', 'model', 'model_seed', 'attack', 'strength', 'attack_seed', 'defense']
    for values, group in attacked.groupby(keys, observed=True):
        aligned = group.merge(clean_frame, on=['example_id', 'gold'], how='left', validate='one_to_one')
        if aligned['clean_prediction'].isna().any():
            raise ValueError('Paired bootstrap could not align clean predictions by example ID')
        row = dict(zip(keys, values))
        row.update(paired_drop_interval(aligned, int(CONFIG['analysis']['bootstrap_resamples']), stable_int(CONFIG['analysis']['bootstrap_seed'], 'paired', *values)))
        row['n'] = len(aligned)
        paired_rows.append(row)
paired_bootstrap = pd.DataFrame(paired_rows)
paired_bootstrap.to_csv(PATHS['tables'] / 'paired_score_drop_bootstrap.csv', index=False)
len(seed_metrics), len(summary), len(bootstrap), len(paired_bootstrap)


## Interpretation

A recovery value of 1 means the defense restores the raw clean-to-attacked loss on average; values below 0 indicate further harm and values above 1 indicate over-recovery. Recovery is undefined when the raw attack causes no score drop. Confidence intervals should be read with condition coverage and seed variation.